# Notebook 06 — Performance & Best Practices

**numpy-mastery** · Module 06 of 06

> **Goal**: understand *why* NumPy is fast, learn to recognise and eliminate
> performance anti-patterns, and use memory layout, stride tricks, and file I/O
> correctly.

**What you'll be able to do after this notebook:**
- Explain the performance difference between NumPy and plain Python at the hardware level
- Identify and fix the four most common NumPy anti-patterns
- Use stride tricks to build sliding-window views with zero copies
- Save and load arrays efficiently
- Profile NumPy code to find bottlenecks

---


## 0 · Setup

In [ ]:
import numpy as np
import time

def timer(fn, *args, repeat=5, **kwargs):
    """Run fn(*args) `repeat` times and return the best time in ms."""
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        times.append((time.perf_counter() - t0) * 1000)
    return min(times)


---
## 1 · Why NumPy is fast — the mental model

Python lists are arrays of *pointers* to arbitrary Python objects.
Every operation on a list element must:
1. Follow the pointer to the object in memory
2. Decode the Python object header to find its type and value
3. Perform the operation in the Python interpreter

NumPy arrays are *contiguous blocks of typed memory* — like C arrays.
Every operation:
1. Goes directly to the data (no pointer chasing)
2. Runs in compiled C/Fortran code (no interpreter overhead)
3. Can use CPU SIMD instructions to process multiple elements per clock cycle

The result is typically **50–200× faster** for numerical work.


In [ ]:
N = 1_000_000

# Python list approach
py_list = list(range(N))
t_list = timer(lambda: [x * 2 for x in py_list])

# NumPy approach
arr = np.arange(N)
t_numpy = timer(lambda: arr * 2)

print(f"Python list : {t_list:.2f} ms")
print(f"NumPy       : {t_numpy:.2f} ms")
print(f"Speedup     : {t_list / t_numpy:.1f}x")


---
## 2 · Views vs copies — memory and performance

We touched on this in Module 02. Now the performance angle:
a **view** is O(1) — it just changes some metadata.
A **copy** is O(n) — it allocates new memory and copies every byte.

For large arrays this difference is enormous.


In [ ]:
big = np.random.default_rng(0).random(10_000_000)

# O(1) — just metadata change
t_view = timer(lambda: big[::2])
# O(n) — allocates new memory
t_copy = timer(lambda: big[::2].copy())
# Fancy index — always O(n)
idx = np.arange(0, len(big), 2)
t_fancy = timer(lambda: big[idx])

print(f"Slice (view) : {t_view:.3f} ms")
print(f"Slice + copy : {t_copy:.2f} ms")
print(f"Fancy index  : {t_fancy:.2f} ms")


### Checking view vs copy

```python
arr.base is None          # True → arr owns its memory (not a view)
arr.base is original      # True → arr is a view of original
np.shares_memory(a, b)    # True → a and b share memory
arr.flags['OWNDATA']      # True → arr owns its data
```


In [ ]:
arr = np.arange(12).reshape(3, 4)

s = arr[1:, ::2]         # slice → view
print("slice is view      :", np.shares_memory(s, arr))

f = arr[[0, 1]]          # fancy → copy
print("fancy is copy      :", not np.shares_memory(f, arr))

r = arr.reshape(2, 6)    # reshape → usually view
print("reshape is view    :", np.shares_memory(r, arr))

t = arr.T                # transpose → view
print("transpose is view  :", np.shares_memory(t, arr))


---
## 3 · The four anti-patterns — and how to fix them

These are the mistakes that silently destroy NumPy performance.


### Anti-pattern 1 — Looping over elements

In [ ]:
arr = np.random.default_rng(0).random(100_000)

# ❌ SLOW: Python loop
def loop_sqrt(a):
    out = np.empty_like(a)
    for i in range(len(a)):
        out[i] = np.sqrt(a[i])
    return out

# ✅ FAST: ufunc
def vec_sqrt(a):
    return np.sqrt(a)

t_loop = timer(loop_sqrt, arr)
t_vec  = timer(vec_sqrt, arr)
print(f"Loop : {t_loop:.2f} ms")
print(f"Ufunc: {t_vec:.3f} ms")
print(f"Speedup: {t_loop/t_vec:.0f}x")


### Anti-pattern 2 — Growing arrays with `np.append`

In [ ]:
# ❌ SLOW: np.append in a loop — O(n²) total allocations
def grow_append(n):
    result = np.array([])
    for i in range(n):
        result = np.append(result, i * 2.0)
    return result

# ✅ FAST: pre-allocate then fill
def grow_preallocate(n):
    result = np.empty(n)
    for i in range(n):
        result[i] = i * 2.0
    return result

# ✅ FASTEST: vectorise entirely
def grow_vectorised(n):
    return np.arange(n, dtype=float) * 2.0

n = 5_000
t_append = timer(grow_append, n)
t_prealloc = timer(grow_preallocate, n)
t_vec = timer(grow_vectorised, n)

print(f"np.append loop  : {t_append:.2f} ms")
print(f"pre-allocate    : {t_prealloc:.2f} ms")
print(f"vectorised      : {t_vec:.3f} ms")


### Anti-pattern 3 — Using Python `math` on arrays

In [ ]:
import math

arr = np.random.default_rng(0).random(100_000)

# ❌ SLOW: math.sqrt only works on scalars; calling it per element is slow
def math_sqrt(a):
    return np.array([math.sqrt(x) for x in a])

# ✅ FAST: np.sqrt handles the whole array at once
def np_sqrt(a):
    return np.sqrt(a)

t_math = timer(math_sqrt, arr)
t_np   = timer(np_sqrt, arr)
print(f"math.sqrt loop: {t_math:.2f} ms")
print(f"np.sqrt       : {t_np:.3f} ms")
print(f"Speedup: {t_math/t_np:.0f}x")


### Anti-pattern 4 — Unnecessary copies

In [ ]:
arr = np.random.default_rng(0).random((1000, 1000))

# ❌ creates an intermediate copy
def with_copy(a):
    b = a.copy()
    b += 1
    return b

# ✅ in-place — no extra allocation (when safe to modify in place)
def in_place(a):
    a = a.copy()   # one copy intentional, then modify in place
    a += 1
    return a

# ✅ out-of-place without intermediate copy
def out_of_place(a):
    return a + 1

t1 = timer(with_copy, arr)
t2 = timer(in_place, arr)
t3 = timer(out_of_place, arr)
print(f"with_copy   : {t1:.2f} ms")
print(f"in_place    : {t2:.2f} ms")
print(f"out_of_place: {t3:.2f} ms")


---
## 4 · Memory layout — C order vs Fortran order

NumPy arrays are stored in **row-major (C) order** by default: elements of
the same row are adjacent in memory. This means iterating along rows
(axis=1) is faster than iterating along columns (axis=0).


In [ ]:
arr = np.random.default_rng(0).random((2000, 2000))

# Row-wise sum — contiguous memory access (cache-friendly)
t_row = timer(lambda: arr.sum(axis=1))
# Column-wise sum — strided memory access (cache-unfriendly for large arrays)
t_col = timer(lambda: arr.sum(axis=0))

print(f"row-wise sum (axis=1): {t_row:.3f} ms")
print(f"col-wise sum (axis=0): {t_col:.3f} ms")

# Strides tell you how many bytes to jump for each step along each axis
print(f"\nstrides: {arr.strides}")
# e.g. (16000, 8): jump 16000 bytes for next row, 8 for next column


### `np.ascontiguousarray` — force C-contiguous layout

After fancy indexing or certain transposes, an array may no longer be
contiguous. This can slow down subsequent operations.


In [ ]:
arr = np.random.default_rng(0).random((1000, 1000))
arr_T = arr.T   # transpose — now Fortran-contiguous, not C-contiguous

print("arr   C-contiguous:", arr.flags['C_CONTIGUOUS'])
print("arr.T C-contiguous:", arr_T.flags['C_CONTIGUOUS'])

# Force C-contiguous copy — often worth it before heavy computation
arr_T_c = np.ascontiguousarray(arr_T)
print("after  C-contiguous:", arr_T_c.flags['C_CONTIGUOUS'])


---
## 5 · Stride tricks — sliding windows with zero copies

`np.lib.stride_tricks.as_strided` lets you create a **view** of an array
with custom strides — the most powerful (and dangerous) tool in NumPy.

**Use case**: sliding window operations (convolution, moving average,
time-series windowing) — without copying any data.

> ⚠️ `as_strided` can read out-of-bounds memory if shapes/strides are wrong.
> Always set `writeable=False` on the result.


In [ ]:
from numpy.lib.stride_tricks import as_strided

arr = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
W = 3   # window size
N = len(arr)

# Without stride tricks: build by indexing (COPIES data)
windows_copy = np.array([arr[i:i+W] for i in range(N - W + 1)])
print("copy approach:\n", windows_copy)

# With stride tricks: zero-copy VIEW
windows_view = as_strided(
    arr,
    shape=(N - W + 1, W),
    strides=(arr.strides[0], arr.strides[0]),  # step 1 element for both dims
    writeable=False
)
print("\nstride trick view:\n", windows_view)
print("shares memory:", np.shares_memory(windows_view, arr))


In [ ]:
# Practical example: rolling mean using stride tricks
arr = np.arange(10, dtype=float)
W = 3

windows = as_strided(
    arr,
    shape=(len(arr) - W + 1, W),
    strides=(arr.strides[0], arr.strides[0]),
    writeable=False
)
rolling_mean = windows.mean(axis=1)
print("array       :", arr)
print("rolling mean:", rolling_mean)
# [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]


---
## 6 · File I/O — saving and loading arrays

| Format | Function | Speed | Portability | Use when |
|--------|----------|-------|-------------|----------|
| `.npy` | `np.save` / `np.load` | ⚡ Fast | NumPy only | single array, internal use |
| `.npz` | `np.savez` / `np.load` | ⚡ Fast | NumPy only | multiple arrays |
| `.npz` compressed | `np.savez_compressed` | 🐢 Slower | NumPy only | storage-constrained |
| `.csv` | `np.savetxt` / `np.loadtxt` | 🐢 Slow | Universal | sharing with non-Python tools |


In [ ]:
import tempfile, os

arr = np.random.default_rng(0).random((500, 200))

with tempfile.TemporaryDirectory() as tmp:

    # .npy — single array
    np.save(f'{tmp}/data.npy', arr)
    loaded = np.load(f'{tmp}/data.npy')
    print("npy round-trip match:", np.array_equal(arr, loaded))
    print("npy file size       :", os.path.getsize(f'{tmp}/data.npy') // 1024, "KB")

    # .npz — multiple arrays
    labels = np.arange(500)
    np.savez(f'{tmp}/dataset.npz', X=arr, y=labels)
    data = np.load(f'{tmp}/dataset.npz')
    print("npz keys            :", list(data.keys()))
    print("npz X match         :", np.array_equal(data['X'], arr))

    # .npz compressed
    np.savez_compressed(f'{tmp}/compressed.npz', X=arr)
    print("compressed size     :", os.path.getsize(f'{tmp}/compressed.npz') // 1024, "KB")

    # .csv — portable but ~10x larger and slower
    np.savetxt(f'{tmp}/data.csv', arr[:5, :5], delimiter=',', fmt='%.6f')
    csv_arr = np.loadtxt(f'{tmp}/data.csv', delimiter=',')
    print("csv round-trip match:", np.allclose(arr[:5, :5], csv_arr))


---
## 7 · Quick profiling with `%timeit` (Jupyter)

In a Jupyter notebook, use the `%timeit` magic for accurate benchmarking.
It automatically runs the expression many times and reports the best result.

```python
arr = np.random.default_rng(0).random(1_000_000)

%timeit np.sqrt(arr)          # ufunc
%timeit arr ** 0.5             # equivalent but slightly slower
%timeit np.sum(arr)
%timeit arr.sum()              # method form — same speed
```

Outside Jupyter, use `time.perf_counter()` like the `timer()` helper
defined in Section 0.


---
## 8 · Best practices — the complete checklist

```
✅ DO
─────────────────────────────────────────────────────────
  Operate on whole arrays, not elements
  Use ufuncs (np.sqrt, np.exp, np.abs ...) over math.*
  Use axis= to aggregate without loops
  Use broadcasting to avoid explicit tiling
  Pre-allocate with np.empty/np.zeros then fill
  Use np.save/.npz for fast array serialisation
  Set writeable=False on as_strided views
  Check np.shares_memory before assuming view/copy
  Use ddof=1 for sample statistics
  Use np.random.default_rng(seed) for reproducibility

❌ AVOID
─────────────────────────────────────────────────────────
  Looping over array elements in Python
  np.append inside a loop (O(n²) allocations)
  math.* functions on arrays
  Unnecessary .copy() calls
  np.random.seed() global state
  Ignoring dtype — float64 uses 2× memory of float32
  Comparing arrays with == for equality (use np.allclose)
```

---
